<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c34-header.png" alt="Nextia Learning · Anomaly Detection: Find What Is Unusual" width="100%">

# Normal changes with the hour and the season

**[Normal changes with the hour and the season](https://learning.nextia-ai.com/courses/anomaly-detection/m05/normal-changes-with-the-hour-and-the-season/)** · Anomaly Detection: Find What Is Unusual · Module 5, lesson 1 of 5 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** see why one threshold for every day fails on a series with weeks, seasons and holidays, and build rolling and seasonal baselines that compare each day with the right "normal".

| | |
|---|---|
| **Time** | About 35 minutes |
| **Needs** | An internet connection for the setup cells (less than 1 MB). No account. |
| **You should know** | Median and MAD, from [Median, MAD and IQR](https://learning.nextia-ai.com/courses/anomaly-detection/m02/median-mad-and-iqr/). The forecasting course explains trend and season in depth; this lesson needs only the idea. |
| **You do not need** | A local project. The setup below downloads everything. |
| **Tested** | Python 3.12.13 in Jupyter on macOS (Apple silicon), from a fresh start (*Restart and run all*), with the package versions of the setup cell |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/anomaly-detection/m05/normal-changes-with-the-hour-and-the-season/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C34/notebooks/M05-L01-normal-changes-with-the-hour-and-the-season/seasonal-baselines-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs the versions that the course uses: pandas 3.1.1, numpy 2.5.4, scipy 1.18.1, scikit-learn 1.9.1, matplotlib 3.11.2. It needs an internet connection. The first time, it takes up to a minute.

You should see one line per package, with the same versions. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again. A restart is needed when an older version was already loaded.

In [ ]:
%pip install -q --disable-pip-version-check pandas==3.1.1 numpy==2.5.4 scipy==1.18.1 scikit-learn==1.9.1 matplotlib==3.11.2
from importlib.metadata import version
for p in ['pandas', 'numpy', 'scipy', 'scikit-learn', 'matplotlib']:
    print(p, version(p))

## Setup: get the course data

Run the next cell. It downloads the data that this lesson uses (`orders.csv`, `tickets_daily.csv`, `calendar.csv`) into a folder `anomaly-course/data`, and checks each file's checksum. It needs an internet connection and no account. It takes a few seconds.

You should see `Ready:` and the files with their sizes. If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `anomaly-course` (in Colab: the **Files** panel on the left), then run the cell again.

The data is made up for this course: no real people, cards or payments. [The dataset card](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C34/data/dataset-card.md) describes every table.

In [ ]:
# Setup: download the course data into anomaly-course/data/ and check each file.
# Standard library only.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

LABS = "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C34/"
FILES = {
    "orders.csv": "bd8059fd73fe294daa157f8a6b6ae3f58319fa644eaa7ff0d31122b02cf05e30",
    "tickets_daily.csv": "36f7067db060fdd2bd084c408c0622370a012773849f243020336f1fc678b31d",
    "calendar.csv": "a5d1d5a3b640a983de72a30fca0e627a4ef23e797b1d80619e9ffcaadf118844",
}
CODE_FILES = []
HERE = Path.cwd()
# The notebook works in its own folder anomaly-course/, marked with a file, so it never writes into another project.
if Path(".nextia-notebook").exists():          # the cell ran before: start again from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
WORK = Path("anomaly-course").resolve()
(WORK / "data").mkdir(parents=True, exist_ok=True)
(WORK / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

def fetch(name, url, path, expected=None):
    """Use a copy next to the notebook if there is one, else download it."""
    if path.exists() and (expected is None or sha256(path) == expected):
        return
    if (HERE / name).exists():
        shutil.copy(HERE / name, path)
    else:
        download(url, path)
    if expected and sha256(path) != expected:
        raise SystemExit(f"{name} has the wrong checksum. Delete the folder anomaly-course and run again.")

for name, expected in FILES.items():
    fetch(name, LABS + "data/" + name, WORK / "data" / name, expected)
for name in CODE_FILES:
    fetch(name, LABS + name, WORK / name)
os.chdir(WORK)
print("Ready:", ", ".join(f"{n} ({(Path('data') / n).stat().st_size / 1e6:.1f} MB)" for n in FILES))

## Setup: helpers

The next cell defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

It also loads pandas as `pd`, NumPy as `np` and matplotlib as `plt`. Run the cell. It prints nothing.

In [ ]:
# Helpers: check an answer without showing it, and the chart style.
import hashlib, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
pd.set_option("display.precision", 3)
pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 20)
warnings.filterwarnings("ignore", category=FutureWarning)
plt.rcParams.update({"figure.figsize": (9, 3.2), "axes.spines.top": False, "axes.spines.right": False})

def fingerprint(value):
    def norm(v):
        if hasattr(v, "to_dict") and hasattr(v, "columns"):
            return {"columns": [str(c) for c in v.columns], "rows": [norm(list(r)) for r in v.itertuples(index=False)]}
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple, set)):
            return [norm(x) for x in (sorted(v, key=str) if isinstance(v, set) else v)]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 4)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. A quiet Thursday or a busy Saturday?

A bakery sells more on Saturday than on Tuesday. If the baker compared every day with the average day, every Saturday would look strange, and a bad Tuesday would look normal. The right question is: is this Saturday unusual **for a Saturday**?

Run the next cell. Two weeks of orders: the second Thursday is low (400). The table shows the **global z-score** (against the mean of all 14 days) and the change against the same day of the week before.

> **Check.** The low Thursday has a global z of -2.16, and the two Saturdays have 1.50 and 1.59.

In [ ]:
days = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"] * 2
orders = np.array([600, 580, 590, 610, 650, 820, 760,    # week 1
                   610, 590, 600, 400, 660, 830, 770])   # week 2: Thursday is low
mean, std = orders.mean(), orders.std(ddof=1)
global_z = (orders - mean) / std
same_day_last_week = np.r_[[np.nan] * 7, orders[:7]]
seasonal_change = orders - same_day_last_week
print(pd.DataFrame({"day": days, "orders": orders, "global z": global_z.round(2),
                    "change vs same day last week": seasonal_change}).to_string(index=False))

Against the mean of all 14 days, the busy Saturdays already look unusual (z up to 1.59), and the low Thursday (-2.16) is not far from them: a global threshold low enough to catch the Thursday also catches the Saturdays. Against the same day of the week before, the Saturdays changed by 10 orders and the Thursday by −210: only the Thursday is strange. A **seasonal baseline** compares each value with the same moment of the cycle: the same weekday, the same hour of the week, the same week of the year.

## 2. Larkfield's daily orders

The next cell loads two years of daily orders (all channels together), the same numbers as in the forecasting course, and the calendar of holidays.

> **Check.** You should see 728 days and 1 missing day.

In [ ]:
orders = pd.read_csv("data/orders.csv", parse_dates=["date"]).groupby("date").orders.sum()
calendar = pd.read_csv("data/calendar.csv", parse_dates=["date"]).set_index("date")
days = pd.date_range(orders.index.min(), orders.index.max(), freq="D")
orders = orders.reindex(days).astype(float)       # a day with no rows becomes NaN (missing), not 0
holiday = calendar.is_holiday.reindex(days).fillna(0).astype(bool)
near_holiday = holiday | holiday.shift(1, fill_value=False) | holiday.shift(-1, fill_value=False)
first_year = orders.loc["2024-04-01":"2025-03-31"]
print(f"{len(orders)} days, mean {orders.mean():.1f} orders a day, {int(orders.isna().sum())} missing day(s)")

## 3. Three baselines

Run the next cell. It builds three detectors and counts their alarm days (after a warm-up of 5 weeks, when every baseline has enough history):

- **global threshold**: more than 3 standard deviations from the mean of the first year;
- **rolling median**: the median and the MAD of the 28 days before the day; alarm when the robust z is above 3.5;
- **seasonal baseline**: the same weekday in the 4 weeks before; alarm when the robust z is above 3.5.

> **Check.** You should see 9 alarm days for the global threshold and 43 for the seasonal baseline.

In [ ]:
def robust_z(x, center, spread):
    return (x - center) / (1.4826 * np.maximum(spread, 1e-9))

alarms = {}
alarms["global threshold"] = (orders - first_year.mean()).abs() > 3 * first_year.std()
med = orders.shift(1).rolling(28, min_periods=14).median()                  # the 28 days BEFORE each day
mad = (orders.shift(1) - med).abs().rolling(28, min_periods=14).median()
alarms["rolling median, 28 days"] = robust_z(orders, med, mad).abs() > 3.5
lags = pd.concat([orders.shift(7 * k) for k in (1, 2, 3, 4)], axis=1)         # the same weekday, 4 weeks
smed = lags.median(axis=1)
smad = lags.sub(smed, axis=0).abs().median(axis=1)
alarms["seasonal baseline, same weekday"] = robust_z(orders, smed, np.maximum(smad, 0.05 * smed)).abs() > 3.5
after_warm_up = orders.index >= "2024-05-06"           # every baseline has 4 weeks of history from here
summary = pd.DataFrame({name: {"alarm days": int((a & after_warm_up).sum()),
                               "on or next to a holiday": int((a & near_holiday & after_warm_up).sum())}
                        for name, a in alarms.items()}).T
summary

The chart shows the seasonal baseline's alarms: orange on or next to a holiday, red on other days.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.4))
ax.plot(orders.index, orders, lw=0.7, color="grey")
a = alarms["seasonal baseline, same weekday"] & after_warm_up
ax.scatter(orders.index[a & near_holiday], orders[a & near_holiday], color="orange", s=16, label="alarm on or next to a holiday", zorder=3)
ax.scatter(orders.index[a & ~near_holiday], orders[a & ~near_holiday], color="red", s=16, label="other alarm", zorder=3)
ax.set_ylabel("orders a day")
ax.legend(loc="upper left")
plt.show()

> **Predict.** Holidays are known in advance: Black Friday, Christmas, Easter. If we mark them as **known events** that are not reported, how many alarm days will each detector have left?

In [ ]:
known_event = near_holiday                               # holidays and the days next to them are not reported
reported = {name: int((a & after_warm_up & ~known_event).sum()) for name, a in alarms.items()}
print(reported)

> **Check.** You should see `{'global threshold': 5, 'rolling median, 28 days': 10, 'seasonal baseline, same weekday': 32}`.

The global threshold reports few days, because one threshold for two years is so wide that it misses almost everything except the biggest days. The seasonal baseline reports many more days, also after the holidays are marked. Some of them are the planted heat-wave week and the shared series' other situations; many are its own noise: four past values are a small reference. The next lessons (STL and forecasts) use more history. A detector is a choice of **what normal is**, and every choice reports different days.

## 4. A common mistake: the missing day

On 2025-10-19 the orders feed sent no rows. Run the next cell.

In [ ]:
print("alarms on 2025-10-19:", {name: bool(a.loc["2025-10-19"]) for name, a in alarms.items()})
print("rows of 2025-10-19:", int(pd.read_csv("data/orders.csv").date.eq("2025-10-19").sum()))

No detector alarms on that day: there is no value to compare. Reading the data with `.reindex()` made the gap visible as `NaN`; reading it with `.groupby()` alone hides it, and some code would even fill it with 0. Count the rows of each day as a separate check: a missing day is a data problem, not a business one.

## 5. Check yourself

**Your turn 1.** How many alarm days does the rolling median have after the warm-up, before marking the holidays? Write it as `rolling_alarms`.

In [ ]:
rolling_alarms = ...   # your code (look at `summary`)

In [ ]:
expect('rolling_alarms', rolling_alarms, 19)

**Your turn 2.** Tickets jumped in one planted week, the heat wave (2025-06-30 to 2025-07-06). Build the same seasonal baseline on the daily tickets (`data/tickets_daily.csv`, all channels together), and list the days of that week it alarms on as strings `"YYYY-MM-DD"` in `heat_wave_alarms`.

In [ ]:
heat_wave_alarms = ...   # your code

In [ ]:
expect_hash('heat_wave_alarms', heat_wave_alarms, '82c2114525d54391')

In the solution the seasonal baseline alarms on 5 of the 7 days. The heat wave is a real event for the help desk: more delivery questions, and the roster did not change.

## Recap

- One global threshold fails on a series with a week, a season and holidays: it is too wide for normal days and too narrow for Black Friday.
- A seasonal baseline asks "unusual for this weekday?"; a rolling baseline follows the level. Mark known events, or they fill the report.
- A missing day is invisible to every detector of values: count the rows.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [STL residuals](https://learning.nextia-ai.com/courses/anomaly-detection/m05/stl-residuals/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/anomaly-detection/m05/normal-changes-with-the-hour-and-the-season/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The practice data is synthetic and has no real people.